# Compare GCE_NFW with Ajello et al. (2016), Figure 13

Copy the **single code cell below** into your analysis notebook after the likelihood fit. It includes the estimated Figure 13 envelope data and needs no separate comparison-data file.

Set `fit_result = joint.results` (or `jl.results`, as appropriate) and check `nfw_path`. The default common region is the paper's **15° × 15° Galactic rectangle**, centered on the GC, including the plane. To compare over 40° × 40°, change `comparison_roi` to `(-20, 20, -20, 20)`.

Let $T$ be the unit-integral NFW-squared FITS map and $f_R=\int_R T\,d\Omega$. The cell applies

$$F_{\rm yours,R}=f_R F_{\rm yours,footprint},\qquad
F_{\rm paper,R}=\frac{f_R}{f_{15\times15}}F_{\rm paper,15\times15}.$$

This avoids confusing your LAT fitting ROI with the spatial footprint over which your spectral amplitude is normalized. Area ratios would be incorrect. The scaling assumes the same standard gamma=1 NFW annihilation morphology; the overall local density cancels. For any portion outside the data fitted in either analysis, this conversion extrapolates the assumed morphology, rather than refitting that region. The actual fitted ROI does not need to be entered because your spectral amplitude already refers to the full unit-integral template.

The four paper envelopes are **graphical estimates** extracted from the hatching in the supplied published PDF (page 19), sampled at 41 logarithmic energies from 1.2–85 GeV where visible. They are accurate to roughly several percent, with larger errors near steep or clipped boundaries, and are suitable for a visual comparison. They are not author-supplied data or four independently reconstructed best-fit central curves. Missing lower edges are explicitly marked NaN; no unplotted tails are invented.

Your orange band is a **pointwise 16th–84th percentile band**, using aligned threeML samples so correlations among amplitude, index, and cutoff are preserved. For maximum likelihood, this is a covariance approximation; for Bayesian results, a posterior interval. Figure 13's caption describes envelopes from normalization and index uncertainties, so the definitions of the bands differ. Differences among the four IEMs also indicate foreground-model sensitivity, which your statistical band alone does not include.

The axes use the paper's units: MeV for energy and MeV cm^-2 s^-1 for E²dN/dE. Your per-keV differential spectrum is converted by dN/dE_MeV = 1000 dN/dE_keV.

**Validation:** executed with synthetic aligned fit samples and a unit-integral test map; checked solid-angle aperture integrals and rescaling for 15° × 15° and 40° × 40°. The paper-band overlay was visually checked against Figure 13. Your actual fitted results and NFW map were not supplied, so the cell has not been executed against your analysis.

Reference: Ajello et al., 2016, *ApJ* **819**, 44, Figure 13, https://doi.org/10.3847/0004-637X/819/1/44.


In [ ]:
# Paste this entire cell AFTER the maximum-likelihood fit.
# Requires numpy, matplotlib, astropy, and your existing threeML fit results.
# Reference: Ajello et al. 2016, ApJ 819, 44, Figure 13 (NOT Figure 18).
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord, skycoord_to_pixel
from astropy.coordinates import SkyCoord
import astropy.units as u

# ---- Settings: change only these lines as needed --------------------------
fit_result = joint.results       # use jl.results if your fit object is jl
nfw_path = "DM_Model/nfw_gamma1.fits"
# Common plotting region in GALACTIC coordinates, with NO plane mask.
# Default: the paper's 15 x 15 degree region. For 40 x 40 use (-20,20,-20,20).
comparison_roi = (-7.5, 7.5, -7.5, 7.5)  # lmin,lmax,bmin,bmax in degrees
boundary_subpixels = 4            # supersampling for fractional boundary pixels
max_samples = 5000
# -------------------------------------------------------------------------

# Read the SAVED best fit, so later changes to the live model do not affect it.
_fitted = fit_result.optimized_model
_shape = _fitted.GCE_NFW.spectrum.main.shape
_spatial = _fitted.GCE_NFW.spatial_shape
if _spatial.K.free or not np.isclose(_spatial.K.value, 1.):
    raise ValueError("This cell assumes the spatial K is fixed at 1.")
if _shape.piv.free:
    raise ValueError("This cell assumes a fixed spectral pivot.")

# Integrate the actual FITS map. The native WCS may be Galactic or equatorial;
# pixel_to_skycoord handles the conversion before testing Galactic ROI cuts.
with fits.open(nfw_path) as _hdul:
    _hdu = next(h for h in _hdul if h.data is not None and h.data.ndim == 2)
    _map = np.asarray(_hdu.data, dtype=float).copy()
    _wcs = WCS(_hdu.header).celestial
if not np.all(np.isfinite(_map)) or np.any(_map < 0):
    raise ValueError("Expected a finite, nonnegative NFW intensity map.")
_ny, _nx = _map.shape
_y, _x = np.indices(_map.shape, dtype=float)

# Exact spherical solid angles of parent pixels, split into two triangles.
def _unit_vectors(x, y):
    return np.moveaxis(pixel_to_skycoord(x, y, _wcs).cartesian.xyz.value, 0, -1)
def _triangle_area(a, b, c):
    numerator = np.abs(np.sum(a * np.cross(b, c), axis=-1))
    denominator = (1 + np.sum(a*b, axis=-1) + np.sum(b*c, axis=-1)
                   + np.sum(c*a, axis=-1))
    return 2 * np.arctan2(numerator, denominator)
_a = _unit_vectors(_x-.5, _y-.5)
_b = _unit_vectors(_x+.5, _y-.5)
_c = _unit_vectors(_x+.5, _y+.5)
_d = _unit_vectors(_x-.5, _y+.5)
_omega = _triangle_area(_a, _b, _c) + _triangle_area(_a, _c, _d)
_total = np.sum(_map * _omega)
if not np.isfinite(_total) or _total <= 0:
    raise ValueError("Invalid template integral.")
if not np.isclose(_total, 1., rtol=.02):
    raise ValueError(f"FITS integral is {_total:g}, expected ~1 sr-weighted. "
                     "Check that this is the unit-integral map used in the fit.")

def _roi_fraction(roi):
    lmin,lmax,bmin,bmax = roi
    if not (-180 <= lmin < lmax <= 180 and -90 < bmin < bmax < 90):
        raise ValueError("Use increasing Galactic bounds in [-180,180].")
    # Ensure requested comparison region is contained in the map footprint.
    ll,bb = np.meshgrid(np.linspace(lmin,lmax,121), np.linspace(bmin,bmax,121))
    xx,yy = skycoord_to_pixel(SkyCoord(ll*u.deg,bb*u.deg,frame='galactic'),_wcs)
    if np.any(~np.isfinite(xx+yy)) or np.any((xx<-.5)|(xx>_nx-.5)|
                                                       (yy<-.5)|(yy>_ny-.5)):
        raise ValueError("Requested ROI extends beyond the NFW template footprint.")
    occupancy = np.zeros_like(_map)
    offsets = (np.arange(boundary_subpixels)+.5)/boundary_subpixels-.5
    for dy in offsets:
        for dx in offsets:
            sky = pixel_to_skycoord(_x+dx,_y+dy,_wcs).galactic
            lon = sky.l.wrap_at(180*u.deg).deg
            lat = sky.b.deg
            occupancy += ((lon>=lmin)&(lon<=lmax)&(lat>=bmin)&(lat<=bmax))
    occupancy /= boundary_subpixels**2
    return np.sum(_map*_omega*occupancy)/_total

_paper_roi = (-7.5,7.5,-7.5,7.5)
_fpaper = _roi_fraction(_paper_roi)
_ftarget = _fpaper if tuple(comparison_roi)==_paper_roi else _roi_fraction(comparison_roi)
if min(_fpaper,_ftarget) <= 0:
    raise ValueError("Empty ROI integral.")
_paper_scale = _ftarget/_fpaper
print(f"Template flux fraction in paper ROI: {_fpaper:.5f}")
print(f"Template flux fraction in comparison ROI: {_ftarget:.5f}")
print(f"Multiply your full-template spectrum by {_ftarget:.5f}.")
print(f"Multiply Figure 13 fluxes by {_paper_scale:.5f}.")

# Correlated samples supplied by threeML: SAME row selection for every parameter.
# For MLE these approximate the likelihood using its fitted covariance; for
# Bayesian results they are posterior samples. No independent parameter draws.
_pars = [_shape.K, _shape.index, _shape.xc]
_columns = [np.asarray(fit_result.get_variates(p.path),dtype=float) if p.free
            else None for p in _pars]
_lengths = [len(a) for a in _columns if a is not None]
if not _lengths or len(set(_lengths)) != 1:
    raise ValueError("Need aligned threeML samples for free GCE parameters.")
_n = _lengths[0]
_rows = np.random.default_rng(2026).choice(_n,min(_n,max_samples),replace=False)
_draws = np.column_stack([a[_rows] if a is not None else
                         np.full(len(_rows),p.value) for a,p in zip(_columns,_pars)])
_valid = np.all(np.isfinite(_draws),axis=1)&(_draws[:,0]>0)&(_draws[:,2]>0)
if not np.all(_valid):
    warnings.warn(f"Discarded {np.mean(~_valid):.1%} nonphysical draws. "
                  "If appreciable, use a profile likelihood or posterior band.")
_draws = _draws[_valid]
if len(_draws)<100:
    raise ValueError("Too few usable uncertainty samples.")

_E_MeV = np.geomspace(1e3,1e5,250)
_E_keV = 1e3*_E_MeV
_pivot = _shape.piv.value
_K,_index,_xc = _draws.T
# Convert dN/dE per keV to per MeV by multiplying by 1000.
# Thus E_MeV^2 * [1000*dN/dE_keV] gives MeV cm^-2 s^-1.
_convert = _ftarget * 1e3 * _E_MeV**2
_best = _convert*np.asarray(_shape(_E_keV),dtype=float)
_expected = (_shape.K.value*(_E_keV/_pivot)**_shape.index.value
             *np.exp(-_E_keV/_shape.xc.value))
if not np.allclose(np.asarray(_shape(_E_keV)),_expected,rtol=1e-6,atol=0):
    raise ValueError("Spectrum is not the assumed Cutoff_powerlaw convention.")
_samples = (_convert[None,:]*_K[:,None]
            *(_E_keV[None,:]/_pivot)**_index[:,None]
            *np.exp(-_E_keV[None,:]/_xc[:,None]))
_low,_high = np.percentile(_samples,[16,84],axis=0)

# Embedded estimates of Fig.13 ENVELOPE EDGES, obtained from the published
# PDF's vector hatching. Columns: E[MeV], lower and upper E^2 dN/dE[MeV/cm2/s]
# for its 15x15 deg ROI. Approximate graphical data, NOT author-supplied tables.
# Hatching thickness/spacing limits accuracy to several percent (worse near
# steep edges). Black/green stop before the lower edge leaves the plotted range;
# no unseen tails are invented. No published best-fit center is implied.
_fig13 = {
    "red": np.array([
        [1200, 0.00017325, 0.00021024],
        [1334.86, 0.00017897, 0.00022058],
        [1484.89, 0.00018475, 0.00023128],
        [1651.77, 0.00019055, 0.00024227],
        [1837.41, 0.00019628, 0.00025347],
        [2043.91, 0.00020143, 0.00026419],
        [2273.61, 0.00020669, 0.00027532],
        [2529.14, 0.00021173, 0.00028646],
        [2813.38, 0.00021689, 0.00029803],
        [3129.57, 0.00022146, 0.00030908],
        [3481.29, 0.00022607, 0.00032044],
        [3872.55, 0.00022945, 0.00033034],
        [4307.77, 0.00023248, 0.00033989],
        [4791.91, 0.00023558, 0.00034984],
        [5330.46, 0.00023878, 0.00036018],
        [5929.53, 0.00024085, 0.000369],
        [6595.93, 0.00024143, 0.00037569],
        [7337.23, 0.00024111, 0.00038106],
        [8161.84, 0.00023845, 0.00038272],
        [9079.13, 0.0002355, 0.00038387],
        [10099.5, 0.00023019, 0.00038108],
        [11234.6, 0.00022495, 0.00037821],
        [12497.2, 0.00021664, 0.0003699],
        [13901.7, 0.00020842, 0.0003614],
        [15464.1, 0.00019691, 0.00034675],
        [17202, 0.00018514, 0.00033107],
        [19135.3, 0.00017163, 0.00031167],
        [21285.9, 0.00015776, 0.00029093],
        [23678.1, 0.00014272, 0.00026726],
        [26339.3, 0.00012675, 0.00024101],
        [29299.5, 0.00011117, 0.00021463],
        [32592.3, 9.4749e-05, 0.00018574],
        [36255.3, 8.018e-05, 0.00015959],
        [40329.9, 6.4972e-05, 0.00013129],
        [44862.5, 5.2568e-05, 0.00010784],
        [49904.4, 4.016e-05, 8.3637e-05],
        [55513.1, 3.0596e-05, 6.4686e-05],
        [61752, 2.1867e-05, 4.6928e-05],
        [68692.1, 1.5406e-05, 3.3561e-05],
        [76412.2, 1.12e-05, 2.2543e-05],
        [85000, 8.2766e-06, 1.4649e-05],
    ]),
    "blue": np.array([
        [1200, 0.00017758, 0.00020576],
        [1334.86, 0.00018117, 0.00020992],
        [1484.89, 0.00018483, 0.00022494],
        [1651.77, 0.00018856, 0.00022948],
        [1837.41, 0.00020204, 0.0002459],
        [2043.91, 0.00020612, 0.00025085],
        [2273.61, 0.00021028, 0.00026881],
        [2529.14, 0.00021455, 0.00027423],
        [2813.38, 0.00021886, 0.00027976],
        [3129.57, 0.00022328, 0.00029978],
        [3481.29, 0.00022778, 0.00030582],
        [3872.55, 0.00023239, 0.00031199],
        [4307.77, 0.00023706, 0.00033433],
        [4791.91, 0.00024187, 0.00034105],
        [5330.46, 0.00024675, 0.00034795],
        [5929.53, 0.00025172, 0.00035496],
        [6595.93, 0.00025681, 0.00036214],
        [7337.23, 0.00024945, 0.00036945],
        [8161.84, 0.00025448, 0.00037691],
        [9079.13, 0.00024717, 0.00038452],
        [10099.5, 0.00025217, 0.00037348],
        [11234.6, 0.00024493, 0.00038103],
        [12497.2, 0.00023789, 0.0003701],
        [13901.7, 0.00023108, 0.00035947],
        [15464.1, 0.00022445, 0.00034917],
        [17202, 0.00020757, 0.00033915],
        [19135.3, 0.00019195, 0.00032943],
        [21285.9, 0.00018644, 0.00030464],
        [23678.1, 0.00016415, 0.00028172],
        [26339.3, 0.0001518, 0.00026052],
        [29299.5, 0.00013366, 0.00024093],
        [32592.3, 0.00011769, 0.00021212],
        [36255.3, 0.00010361, 0.00017782],
        [40329.9, 8.2696e-05, 0.00014906],
        [44862.5, 6.9321e-05, 0.00013124],
        [49904.4, 5.5324e-05, 0.00010474],
        [55513.1, 4.4155e-05, 7.9594e-05],
        [61752, 3.1944e-05, 6.0477e-05],
        [68692.1, 2.311e-05, 4.5957e-05],
        [76412.2, 1.5919e-05, 3.1654e-05],
        [85000, 1.0964e-05, 2.1804e-05],
    ]),
    "green": np.array([
        [1200, 0.0001075, 0.00015922],
        [1334.86, 0.00011068, 0.00018994],
        [1484.89, 0.00011968, 0.00021573],
        [1651.77, 0.00012322, 0.00024502],
        [1837.41, 0.00012685, 0.00029229],
        [2043.91, 0.0001306, 0.00031608],
        [2273.61, 0.00013446, 0.00035899],
        [2529.14, 0.00013843, 0.00040772],
        [2813.38, 0.0001357, 0.0004409],
        [3129.57, 0.0001397, 0.00047676],
        [3481.29, 0.00013694, 0.00051554],
        [3872.55, 0.00012779, 0.00055749],
        [4307.77, 0.00012526, 0.00057395],
        [4791.91, 0.00011691, 0.00059093],
        [5330.46, 0.00010387, 0.00060837],
        [5929.53, 9.694e-05, 0.00059635],
        [6595.93, 8.2007e-05, 0.00058455],
        [7337.23, 7.2865e-05, 0.00054551],
        [8161.84, 5.8687e-05, 0.00048469],
        [9079.13, 4.7271e-05, 0.00043068],
        [10099.5, 3.6247e-05, 0.00036433],
        [11234.6, 2.7795e-05, 0.0003082],
        [12497.2, 1.9321e-05, 0.00023633],
        [13901.7, 1.343e-05, 0.00017255],
        [15464.1, 1.0299e-05, 0.00011995],
        [17202, 7.8973e-06, 8.3376e-05],
        [19135.3, np.nan, 5.0017e-05],
        [21285.9, np.nan, 3.0004e-05],
        [23678.1, np.nan, 1.8e-05],
        [26339.3, np.nan, 1.0281e-05],
    ]),
    "black": np.array([
        [1200, 0.00013244, 0.00017104],
        [1334.86, 0.00014698, 0.00020347],
        [1484.89, 0.00016311, 0.00023378],
        [1651.77, 0.00017484, 0.0002686],
        [1837.41, 0.00018741, 0.00029809],
        [2043.91, 0.00019403, 0.00033081],
        [2273.61, 0.00020089, 0.00036712],
        [2529.14, 0.00020089, 0.0003801],
        [2813.38, 0.00020089, 0.00040744],
        [3129.57, 0.00019403, 0.00040744],
        [3481.29, 0.00018741, 0.00042182],
        [3872.55, 0.00016887, 0.00040744],
        [4307.77, 0.00015217, 0.0003801],
        [4791.91, 0.00012792, 0.0003425],
        [5330.46, 0.00010753, 0.00030862],
        [5929.53, 8.7312e-05, 0.00025944],
        [6595.93, 6.6137e-05, 0.00020347],
        [7337.23, 4.8388e-05, 0.00015957],
        [8161.84, 3.1901e-05, 0.00011276],
        [9079.13, 2.1031e-05, 7.9686e-05],
        [10099.5, 1.2494e-05, 4.901e-05],
        [11234.6, np.nan, 3.0144e-05],
        [12497.2, np.nan, 1.5584e-05],
        [13901.7, np.nan, 8.0579e-06],
    ]),
}

# Both spectra are now intrinsic fluxes in exactly the same Galactic rectangle.
# Assuming the same standard gamma=1 NFW-squared morphology, this is a pure
# aperture conversion. It does not refit the data or reproduce a different IEM.
_labels = {
    'black': 'Pulsars, index-scaled',
    'green': 'OB stars, index-scaled',
    'red': 'Pulsars, intensity-scaled',
    'blue': 'OB stars, intensity-scaled',
}
_hatches = {'black':'---','green':'\\\\','red':'|||','blue':'///'}
_floor = 7.1e-6*_paper_scale  # lower plotted limit in the original figure
fig, ax = plt.subplots(figsize=(10,7),constrained_layout=True)
for color in ('red','blue','green','black'):
    energy,lo,hi = _fig13[color].T
    lo,hi = lo*_paper_scale,hi*_paper_scale
    # NaN = lower boundary was outside the paper's plotted range.
    # Show only its visible portion, without inventing a faint-energy tail.
    visible_lo = np.where(np.isfinite(lo),lo,_floor)
    ax.fill_between(energy,visible_lo,hi,color=color,alpha=.10)
    ax.fill_between(energy,visible_lo,hi,facecolor='none',edgecolor=color,
                    hatch=_hatches[color],linewidth=0,label=_labels[color])
    ax.plot(energy,lo,color=color,lw=.8,alpha=.6)
    ax.plot(energy,hi,color=color,lw=.8,alpha=.6)
ax.fill_between(_E_MeV,_low,_high,color='darkorange',alpha=.35,
                label='Your fit: pointwise 68% band',zorder=9)
ax.plot(_E_MeV,_best,color='darkorange',lw=2.8,label='Your GCE_NFW best fit',zorder=10)
ax.set(xscale='log',yscale='log',xlim=(1e3,1e5),
       ylim=(_floor,max(8e-4*_paper_scale,1.1*np.nanmax(_high))),
       xlabel='Energy (MeV)',
       ylabel=r'$E^2\,dN/dE\quad[\mathrm{MeV\,cm^{-2}\,s^{-1}}]$')
_lmin,_lmax,_bmin,_bmax = comparison_roi
ax.set_title(f'GCE comparison: l = [{_lmin:g}, {_lmax:g}] deg, '
             f'b = [{_bmin:g}, {_bmax:g}] deg')
ax.grid(which='major',alpha=.18)
ax.legend(fontsize=9,loc='best',title='Ajello et al. (2016), Fig. 13: approximate envelopes')
plt.show()
print("Paper envelopes are graphical estimates; no published central curves "
      "or parameter covariance were reconstructed.")
print("Your band is pointwise and includes K/index/xc correlations. For MLE "
      "it is a covariance approximation, not a profile-likelihood interval.")
# Optional: fig.savefig('GCE_Ajello2016_comparison.pdf',bbox_inches='tight')
